# Final visuals — Methods 02–05

Visualization-only notebook. It recreates and displays every figure used by Methods 02–05, but does not write figures or tables to disk. Method 02 is skipped because it produces a results table rather than a plot. Run the method notebooks first so their validated result tables are available.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SUBMISSION_ROOT = Path.cwd() / 'submission' if (Path.cwd() / 'submission' / 'OECD Data.csv').exists() else (Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd())
sys.path.insert(0, str(SUBMISSION_ROOT.parent))
from submission.code.oecd_audit import INDICATOR_SPECS, load_clean

TABLE_DIR = SUBMISSION_ROOT / 'report' / 'tables'
data = load_clean()
sns.set_theme(style='whitegrid', context='notebook')

def require_tables(*names):
    paths = [TABLE_DIR / name for name in names]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError('Run the corresponding method notebooks first. Missing: ' + ', '.join(missing))
    return [pd.read_csv(path) for path in paths]

def independent_series(frame, code):
    subset = frame.loc[frame.indicator_code.eq(code)].copy()
    subset['plot_time'] = np.where(
        subset.indicator_code.isin(['7_1_DEP', '11_2']),
        subset.independent_period.map({'2008-10': 2009, '2011-13': 2012, '2014-16': 2015,
                                       '2017-19': 2018, '2020-22': 2021, '2023-25': 2024}),
        subset.year,
    )
    return subset.sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')

## Method 03 — Comparator-composition bootstrap and placebo ranking

In [ ]:
bootstrap, placebo = require_tables('material_social_bootstrap_results.csv', 'material_social_placebo_results.csv')
plot_table = bootstrap.merge(placebo[['indicator_code', 'australia_favourable_percentile']], on='indicator_code', validate='one_to_one')
names = {'1_1': 'Household income per person', '2_1': 'Employment rate',
         '7_1_DEP': 'Lack of social support', '11_2': 'Negative affect'}
units = {'1_1': 'USD per person, PPP', '2_1': 'Percentage points',
         '7_1_DEP': 'Percentage points', '11_2': 'Percentage points'}
fig, axes = plt.subplots(4, 1, figsize=(10, 9), constrained_layout=True)
for ax, row in zip(axes, plot_table.itertuples(index=False)):
    left = row.observed_oriented_gap - row.bootstrap_ci_lower
    right = row.bootstrap_ci_upper - row.observed_oriented_gap
    ax.axvline(0, color='#555555', linewidth=1)
    ax.errorbar(row.observed_oriented_gap, 0, xerr=[[left], [right]], fmt='o',
                color='#D55E00', ecolor='#0072B2', elinewidth=3, capsize=4, markersize=8)
    ax.set_yticks([])
    ax.set_title(names[row.indicator_code], loc='left', fontweight='bold')
    ax.set_xlabel(f'Oriented Australia-minus-median gap ({units[row.indicator_code]}; positive favours Australia)')
    ax.text(.99, .82, f'Placebo percentile: {row.australia_favourable_percentile:.1f} | {row.eligible_comparator_country_count} comparators | {row.sensitivity_label}', transform=ax.transAxes, ha='right', va='top', fontsize=9)
fig.suptitle('Method 03: Australia’s common-endpoint gaps', fontsize=16, fontweight='bold')
fig.text(.5, -.01, 'Subtitle: intervals measure sensitivity to comparator-country composition, not survey-sampling uncertainty.', ha='center', fontsize=9)
plt.show()

## Method 04 — Theil–Sen and Kendall trend robustness

In [ ]:
trend, slopes = require_tables('material_social_trend_results.csv', 'method4_country_slope_distribution.csv')
order = ['1_1', '2_1', '7_1_DEP', '11_2']
axis_units = {'1_1': 'US dollars per person, PPP converted', '2_1': 'Percentage of population aged 25–64 years',
              '7_1_DEP': 'Percentage of population aged 15 years or over', '11_2': 'Percentage of population aged 15 years or over'}

# Method 04 figure 1: Australian observations and fitted Theil–Sen trends.
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, code in zip(axes.flat, order):
    series = independent_series(data.loc[data.country_code.eq('AUS')], code)
    fit = slopes.loc[slopes.indicator_code.eq(code) & slopes.country_code.eq('AUS')].iloc[0]
    result = trend.loc[trend.indicator_code.eq(code)].iloc[0]
    x = series.plot_time.astype(float).to_numpy(); y = series.value.to_numpy()
    ax.plot(x, y, marker='o', linewidth=1.8, color='#4C78A8')
    ax.plot(x, fit.native_slope_per_year * x + fit.theilsen_intercept, '--', linewidth=2, color='#D55E00', label='Theil–Sen fit')
    ax.set_title(result.short_name, loc='left', fontweight='bold')
    ax.set_ylabel(axis_units[code]); ax.set_xlabel('Year / pooled-period midpoint')
    ax.text(.02, .96, f'Native slope: {result.native_slope_per_year:.3g} per year\nFavourable-scale tau: {result.kendall_tau_favourable:.2f} | n={int(result.n_independent)}', transform=ax.transAxes, va='top', fontsize=8, bbox={'facecolor': 'white', 'alpha': .85, 'edgecolor': 'none'})
    ax.legend(frameon=False, loc='lower right')
fig.suptitle('Method 04A: Australia’s robust direction of change', fontsize=16, fontweight='bold')
fig.text(.5, -.01, 'Subtitle: Theil–Sen slopes use independent observations; pooled social estimates are counted once per window.', ha='center', fontsize=9)
plt.show()

# Method 04 figure 2: Australia within same-span country slope distributions.
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
for ax, code in zip(axes.flat, order):
    group = slopes.loc[slopes.indicator_code.eq(code)].copy()
    peers = group.loc[~group.is_australia]
    australia = group.loc[group.is_australia].iloc[0]
    result = trend.loc[trend.indicator_code.eq(code)].iloc[0]
    sns.boxplot(data=peers, x='favourable_slope_per_year', ax=ax, color='#D9E2EA', width=.35, showfliers=False)
    sns.stripplot(data=peers, x='favourable_slope_per_year', ax=ax, color='#667788', alpha=.6, size=5, jitter=.12)
    ax.scatter(australia.favourable_slope_per_year, 0, marker='D', s=100, color='#D55E00', edgecolor='white', zorder=5, label='Australia')
    ax.axvline(0, color='#333333', linewidth=1, linestyle=':')
    ax.set_title(result.short_name, loc='left', fontweight='bold'); ax.set_yticks([])
    ax.set_xlabel(f'Favourable-oriented slope ({result.native_unit_per_year})')
    ax.text(.02, .95, f'Australia percentile: {result.australia_favourable_slope_percentile:.0f}\nComparators: {int(result.n_comparator_countries)}', transform=ax.transAxes, va='top', fontsize=9, bbox={'facecolor': 'white', 'alpha': .88, 'edgecolor': '#CCCCCC'})
    ax.legend(frameon=False, loc='lower right')
fig.suptitle('Method 04B: Australia relative to same-span country trend distributions', fontsize=16, fontweight='bold')
fig.text(.5, -.01, 'Subtitle: favourable-oriented slopes align higher-is-better and lower-is-better outcomes.', ha='center', fontsize=9)
plt.show()

## Method 05 — Spearman association

In [ ]:
spearman, = require_tables('material_social_spearman_results.csv')
endpoints = {'1_1': (2010, 2024), '2_1': (2010, 2024), '7_1_DEP': (2010, 2024), '11_2': (2010, 2024)}
def oriented_changes(code):
    start, end = endpoints[code]
    subset = data.loc[data.indicator_code.eq(code) & data.year.isin([start, end])].copy()
    subset = subset.sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
    wide = subset.pivot(index='country_code', columns='year', values='value').reindex(columns=[start, end]).dropna().reset_index()
    sign = 1 if INDICATOR_SPECS[code].direction == 'higher' else -1
    wide['change'] = (wide[end] - wide[start]) * sign
    return wide[['country_code', 'change']]
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax, row in zip(axes.flat, spearman.itertuples(index=False)):
    material = oriented_changes(row.material_indicator_code).rename(columns={'change': 'material'})
    social = oriented_changes(row.social_indicator_code).rename(columns={'change': 'social'})
    paired = material.merge(social, on='country_code')
    peers = paired.loc[paired.country_code.ne('AUS')]
    aus = paired.loc[paired.country_code.eq('AUS')].iloc[0]
    ax.scatter(peers.material, peers.social, color='#7A7A7A', alpha=.7, s=34, label='Eligible comparators')
    ax.scatter(aus.material, aus.social, marker='D', color='#D55E00', edgecolor='black', linewidth=.5, s=72, label='Australia', zorder=3)
    ax.axhline(0, color='#555555', linewidth=.8); ax.axvline(0, color='#555555', linewidth=.8)
    ax.set_title(f'{row.material_outcome} and {row.social_outcome}', loc='left', fontweight='bold')
    ax.set_xlabel(f'Favourable {row.material_outcome.lower()} change'); ax.set_ylabel(f'Favourable {row.social_outcome.lower()} change')
    ax.text(.02, .98, f'Spearman ρ = {row.spearman_rho:.2f}\n95% CI [{row.bootstrap_ci_lower:.2f}, {row.bootstrap_ci_upper:.2f}]\nn = {row.comparator_country_count}', transform=ax.transAxes, va='top', fontsize=9, bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': .85})
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle('Method 05: Exploratory cross-country associations', fontsize=16, fontweight='bold')
fig.text(.5, -.01, 'Subtitle: positive oriented values indicate improvement; Australia is displayed but excluded from association estimates.', ha='center', fontsize=9)
plt.show()